# PhenoAssistant Case 3 on Microsoft Agent Framework

CPU-safe migration of `case3.ipynb`.

The original Case 3 workflow covers winter-wheat nutrient-deficiency image
classification, dataset preparation guidance, fine-tuning, and model-zoo
inspection.

The current repository is already in a post-training state: the expected
winter-wheat DINOv2 LoRA checkpoint is registered in `model_zoo.json`.

This notebook therefore preserves current repository truth rather than
pretending the model is absent or retraining it.

The CPU migration exposes two bounded capabilities:

- read-only model-zoo discovery;
- Case 3 training/inference readiness assessment.

Actual dataset upload, model loading, training, and inference remain deferred
to the canonical Nottingham GPU environment.

In [ ]:
from __future__ import annotations

import inspect
import os
from pathlib import Path
from typing import Any

from phenoassistant_maf import (
    OpenRouterSettings,
    build_application,
    create_chat_client,
    run_application,
)

ROOT = Path.cwd()

BASE_DATA = ROOT / "results/demo/potato_phenotypes.csv"
MODEL_ZOO = ROOT / "model_zoo.json"

CASE3_DATASET = (
    ROOT
    / "data"
    / "winter-wheat_nutri-defi-identify_dndww20"
)

EXPECTED_CHECKPOINT = (
    "fengchen025/"
    "winter-wheat_nutri-defi-identify_dndww20_dino2b_lora"
)

assert BASE_DATA.is_file()
assert MODEL_ZOO.is_file()

print("MODEL_ZOO=", MODEL_ZOO.relative_to(ROOT))
print("CASE3_DATASET_EXISTS=", CASE3_DATASET.is_dir())
print("EXPECTED_CHECKPOINT=", EXPECTED_CHECKPOINT)

## Original training and inference operations — explicit GPU boundary

The legacy Case 3 implementation can prepare/upload a Hugging Face dataset,
fine-tune an image classifier, load a checkpoint, and run inference.

Those operations are deliberately not executed in this CPU migration.

The migrated workflow instead answers three bounded questions:

1. Which image-classification checkpoint is currently registered?
2. Can the Case 3 training workflow begin from the current repository state?
3. Is the registered classifier at the project-defined inference-readiness
   boundary?

The readiness capability also preserves the legacy training contract,
including the supported `lora` and `fullft` fine-tuning methods, without
performing a mutation or GPU operation.

In [ ]:
if not os.environ.get("OPENROUTER_API_KEY"):
    raise RuntimeError("Set OPENROUTER_API_KEY before running this cell.")

os.environ.setdefault("OPENROUTER_MODEL", "openrouter/free")

settings = OpenRouterSettings.from_env()
client = create_chat_client(settings)


def forbidden_calculator(
    a: int,
    b: int,
    operator: str,
) -> int:
    raise AssertionError(
        "Case 3 selected calculator unexpectedly."
    )


def forbidden_anova(
    data_path: str,
    descriptor: str,
    within_subject_factor: str,
    between_subject_factor: str,
    subject_id: str,
    save_path: str | None = None,
):
    raise AssertionError(
        "Case 3 selected ANOVA unexpectedly."
    )


def forbidden_tukey(
    data_path: str,
    descriptor: str,
    between_subject_factor: str,
    subject_id: str,
    save_path: str | None = None,
):
    raise AssertionError(
        "Case 3 selected Tukey unexpectedly."
    )


def forbidden_regression(
    data_path: str,
    x_column: str,
    y_column: str,
    plot_path: str,
):
    raise AssertionError(
        "Case 3 selected regression unexpectedly."
    )


def forbidden_aggregate(
    data_path: str,
    operation: str,
    value_column: str,
    filter_column: str | None,
    filter_value: str | None,
):
    raise AssertionError(
        "Case 3 selected CSV aggregation unexpectedly."
    )


application = build_application(
    client=client,
    data_path=str(BASE_DATA),
    calculator_callable=forbidden_calculator,
    anova_callable=forbidden_anova,
    tukey_callable=forbidden_tukey,
    regression_callable=forbidden_regression,
    aggregate_callable=forbidden_aggregate,
    model_zoo_path=str(MODEL_ZOO),
    case3_dataset_path=str(CASE3_DATASET),
)

print("MODEL=", settings.model)
print("TOOLS=", application.registry.names)

In [ ]:
def response_text(response: Any) -> str:
    messages = getattr(
        response,
        "messages",
        None,
    )

    if messages:
        text = getattr(
            messages[-1],
            "text",
            None,
        )

        if isinstance(text, str):
            return text.strip()

    return str(response).strip()

In [ ]:
# Original model-zoo inspection intent.
model_response = await run_application(
    application,
    '''
    What image-classification models are currently registered in the
    PhenoAssistant vision model zoo?

    Call get_model_catalogue exactly once for image-classification.
    Do not load or execute any model.
    Summarise only returned evidence.
    ''',
)

print(response_text(model_response))

In [ ]:
# Original training / fine-tuning intent represented as a readiness boundary.
training_response = await run_application(
    application,
    '''
    Assess whether the Case 3 winter-wheat image-classification training
    workflow can begin from the current repository state.

    Call assess_case3_readiness exactly once for training.

    Report the dataset readiness evidence, registered checkpoint state,
    supported fine-tuning methods, and blockers.

    Do not train, upload data, load a model, or modify Hugging Face.
    Summarise only returned evidence.
    ''',
)

print(response_text(training_response))

In [ ]:
# Registered-model inference readiness.
inference_response = await run_application(
    application,
    '''
    Assess the current Case 3 winter-wheat classifier for inference readiness.

    Call assess_case3_readiness exactly once for inference.

    Report whether the expected checkpoint is registered, whether the
    registered-model preconditions are satisfied, and the current execution
    boundary.

    Do not load the checkpoint or run inference.
    Summarise only returned evidence.
    ''',
)

print(response_text(inference_response))

In [ ]:
print("CASE3_MAF_CPU_WORKFLOW_COMPLETE")


async def close_provider(value: Any) -> None:
    for candidate in (
        value,
        getattr(value, "client", None),
        getattr(value, "_client", None),
        getattr(value, "_openai_client", None),
    ):
        if candidate is None:
            continue

        for name in (
            "aclose",
            "close",
        ):
            method = getattr(
                candidate,
                name,
                None,
            )

            if callable(method):
                result = method()

                if inspect.isawaitable(result):
                    await result

                return


await close_provider(client)

print("OPENROUTER_CLIENT_CLOSED")